# PREPROCESADO TEMPORAL

Preprocesado creado de manera temporal debido a que en caso de que estén documentando el código no generar problemas con el github

Una vez se sepa que no genera problemas simplemente copiar la información que está en este preprocesado en el que ya se tiene, es decir, "02_Preprocesado_y_Split".

Esto se inserta al final del primer preprocesado.

# Carga de Librerías

In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import LabelEncoder

# Carga de Datos

In [3]:
dengue = pd.read_csv(
    "../data/raw/RNN/vectoria_dengue_semanal.csv"
)

acciones = pd.read_csv(
    "../data/raw/Acciones/vectoria_acciones.csv"
)

# 11. Preprocesamiento Modelo 2

Ahora se creará el preprocesamiento del modelo 2, esto para poseer lo necesario para comenzar con el modelo como tal

## 11.1 Conversión de Fechas

In [4]:
dengue["fecha_inicio_semana"] = pd.to_datetime(
    dengue["fecha_inicio_semana"]
)

dengue = dengue.sort_values(
    "fecha_inicio_semana"
)

dengue.reset_index(
    drop=True,
    inplace=True
)

C:\Users\sebas\AppData\Local\Temp\ipykernel_23084\124348772.py:9: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  dengue.reset_index(


Se convierten las fechas de manera explícita como una fecha, lo cual a la hora de graficar el eje temporal o filtros por rangos de fecha estos se harán de manera correcta.

## 11.2 Selección de Variables

In [5]:
modelo2 = dengue[
    [
        "fecha_inicio_semana",
        "casos_dengue",
        "lluvia_mm",
        "temp_max_c",
        "temp_min_c",
        "humedad_rel_pct"
    ]
].copy()

Se seleccionan las variables con las que se van a trabajar.

## 11.3 Verificación de Nulos

In [6]:
modelo2.isnull().sum()

fecha_inicio_semana     0
casos_dengue            0
lluvia_mm              35
temp_max_c             34
temp_min_c              0
humedad_rel_pct        29
dtype: int64

Al haber nulos se realizaráuna interpolación temporal, la interpolación temporal es un valor intermedio que se realiza basado en una línea recta que conecta con el valor anterior y el valor posterior, permitiendo obtener un valor proporcional al de la semana anterior y la que le sigue, evitando borrar una fila completa.

In [7]:
modelo2 = modelo2.interpolate(
    method="linear"
)

modelo2.isnull().sum()

fecha_inicio_semana    0
casos_dengue           0
lluvia_mm              0
temp_max_c             0
temp_min_c             0
humedad_rel_pct        0
dtype: int64

# 12 Variables Rezagadas

Crea variables con retraso para permitir que el modelo de series temporal pueda predecir dentro de cuatro semanas

## 12.1 Casos

In [8]:
for lag in [1, 2, 3, 4]:

    modelo2[f"casos_lag_{lag}"] = (
        modelo2["casos_dengue"]
        .shift(lag)
    )

## 12.2 Lluvia

In [9]:
for lag in [1, 2, 3, 4]:

    modelo2[f"lluvia_lag_{lag}"] = (
        modelo2["lluvia_mm"]
        .shift(lag)
    )

## 12.3 Temperatura Máxima

In [10]:
for lag in [1, 2, 3, 4]:

    modelo2[f"temp_max_lag_{lag}"] = (
        modelo2["temp_max_c"]
        .shift(lag)
    )

## 12.4 Temperatura Mínima

In [11]:
for lag in [1, 2, 3, 4]:

    modelo2[f"temp_min_lag_{lag}"] = (
        modelo2["temp_min_c"]
        .shift(lag)
    )

## 12.5 Humedad

In [12]:
for lag in [1, 2, 3, 4]:

    modelo2[f"humedad_lag_{lag}"] = (
        modelo2["humedad_rel_pct"]
        .shift(lag)
    )

# 13. Variable Objeto

In [13]:
modelo2["target"] = (
    modelo2["casos_dengue"]
    .shift(-4)
)

En este caso la variable objeto es "casos_dengue" debido a que esto es lo que queremos predecir con las variables rezagadas

# 14. Eliminación de Nulos

In [14]:
modelo2 = modelo2.dropna()

modelo2.reset_index(
    drop=True,
    inplace=True
)

C:\Users\sebas\AppData\Local\Temp\ipykernel_23084\797612187.py:3: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  modelo2.reset_index(


Este código permite eliminar nulos que se generan al inicio y al final

## 15.1 Escalado

In [15]:
scaler = MinMaxScaler()

columnas = modelo2.columns.drop(
    "fecha_inicio_semana"
)

modelo2[columnas] = (
    scaler.fit_transform(
        modelo2[columnas]
    )
)

In [16]:
modelo2.head()

,fecha_inicio_semana,casos_dengue,lluvia_mm,temp_max_c,temp_min_c,humedad_rel_pct,casos_lag_1,casos_lag_2,casos_lag_3,casos_lag_4,...,temp_max_lag_4,temp_min_lag_1,temp_min_lag_2,temp_min_lag_3,temp_min_lag_4,humedad_lag_1,humedad_lag_2,humedad_lag_3,humedad_lag_4,target
0,2014-01-05,0.061584,0.000000,0.388889,0.392,0.419753,0.049853,0.102639,0.020528,0.079179,...,0.444444,0.784,1.000000,0.809524,0.507937,0.938272,0.421811,0.267490,0.218107,0.120235
1,2014-01-05,0.000000,0.060550,0.177778,0.048,0.617284,0.061584,0.049853,0.102639,0.020528,...,0.761111,0.392,0.777778,1.000000,0.809524,0.419753,0.938272,0.421811,0.267490,0.026393
2,2014-01-05,0.008798,0.002359,0.361111,0.296,0.462963,0.000000,0.061584,0.049853,0.102639,...,0.672222,0.048,0.388889,0.777778,1.000000,0.617284,0.419753,0.938272,0.421811,0.017595
3,2014-01-12,0.026393,0.000000,0.655556,0.864,0.423868,0.008798,0.000000,0.061584,0.049853,...,0.594444,0.296,0.047619,0.388889,0.777778,0.462963,0.617284,0.419753,0.938272,0.000000
4,2014-01-12,0.120235,0.003670,0.555556,0.792,0.748971,0.026393,0.008798,0.000000,0.061584,...,0.388889,0.864,0.293651,0.047619,0.388889,0.423868,0.462963,0.617284,0.419753,0.043988


Se escalan los datos debido a que RNN y LSTM funcionan mejor con valores entre 0 y 1.

# 16. Guardar Archivo Procesado

In [17]:
modelo2.to_csv(
    "../data/processed/modelo2_preprocesado.csv",
    index=False
)

# 17. Preprocesamiento Modelo 3

Ahora se realizará el preprocesamiento del modelo 3 el cual tiene como objetivo dar que medidas de prevencion dar según las condiciones que se le den.

## 17.1 Selección de Variables

In [25]:
modelo3 = acciones[
    [
        "casos_pred_4sem",
        "casos_umbral_endemico",
        "lluvia_14d_mm",
        "temp_max",
        "temp_min",
        "humedad",
        "aegypti_detectado",
        "recomendacion"
    ]
].copy()

## 17.2 Verificación de Nulos

In [26]:
modelo3.isnull().sum()

casos_pred_4sem          0
casos_umbral_endemico    0
lluvia_14d_mm            0
temp_max                 0
temp_min                 0
humedad                  0
aegypti_detectado        0
recomendacion            0
dtype: int64

Observamos que no posee valores faltantes.

# 18. Codificaicón de Objetivo

In [28]:
encoder = LabelEncoder()

modelo3["recomendacion"] = (
    encoder.fit_transform(
        modelo3["recomendacion"]
    )
)

Esto nos permite convertir la variable "recomendacion" a una variable numérica, con el objetivo de que el modelo pueda comprender dicha variable.

# 19. Mapeo

In [29]:
dict(
    zip(
        encoder.classes_,
        encoder.transform(
            encoder.classes_
        )
    )
)

{'alerta_sanitaria': np.int64(0),
 'eliminacion_criaderos': np.int64(1),
 'fumigacion_focalizada': np.int64(2),
 'monitoreo_normal': np.int64(3)}

Este bloque nos permite generar un diccionario para mapear el número que le fue asignado a cada categoría de texto después de convertirlas en número en el bloque anterior.

# 20. Escalado

In [30]:
feature_cols = modelo3.columns.drop(
    "recomendacion"
)

scaler_modelo3 = MinMaxScaler()

modelo3[feature_cols] = (
    scaler_modelo3.fit_transform(
        modelo3[feature_cols]
    )
)

De igual forma, se escala para que el modelo pueda tener un mejor entendimiento de los datos que hay en el dataset

# 21. Guardado de los Datos

In [31]:
modelo3.to_csv(
    "../data/processed/modelo3_preprocesado.csv",
    index=False
)